# 02. Geração de Pseudo-Labels com DeepForest

Este notebook executa o modelo pré-treinado DeepForest (RetinaNet) sobre as imagens compactadas no arquivo HDF5, converte as caixas para o formato YOLO [class_id, x_center, y_center, w, h] normalizado, e salva as anotações geradas no subgrupo `labels` correspondente.

**Responsável:** Pessoa 3

In [10]:
import sys
sys.path.append('..')

import utils
import h5py
import numpy as np
import cv2

In [11]:
import h5py

HDF5_PATH = "dataset_v1_raw.h5"

hdf5 = h5py.File(HDF5_PATH, "r")
images = hdf5["images"]

print("OK carregado:", len(images))

OK carregado: 0


## 1. Carregamento do HDF5 e do Modelo DeepForest

In [12]:
class FakeModel:
    def predict_image(self, image, return_plot=False):
        import pandas as pd

        return pd.DataFrame([
            {
                "xmin": 100,
                "ymin": 100,
                "xmax": 300,
                "ymax": 300,
                "score": 0.95
            }
        ])

model = FakeModel()

print("DeepForest MOCK ativo (ambiente liberado)")

DeepForest MOCK ativo (ambiente liberado)


## 2. Predição de Caixas e Normalização YOLO (Pessoa 3)

Conversão de [xmin, ymin, xmax, ymax] em pixels para as coordenadas normalizadas [0.0, 1.0].

In [13]:
def convert_to_yolo(box, img_w, img_h):
    """
    box: [xmin, ymin, xmax, ymax]
    retorna: [x_center, y_center, width, height] normalizado
    """
    xmin, ymin, xmax, ymax = box

    x_center = (xmin + xmax) / 2.0 / img_w
    y_center = (ymin + ymax) / 2.0 / img_h
    width = (xmax - xmin) / img_w
    height = (ymax - ymin) / img_h

    return [x_center, y_center, width, height]


pseudo_labels = []

for i in range(len(images)):
    img = images[i]

    if img.dtype != np.uint8:
        img = img.astype(np.uint8)

    preds = model.predict_image(image=img, return_plot=False)

    img_h, img_w = img.shape[:2]

    if preds is not None and len(preds) > 0:
        for _, row in preds.iterrows():
            box = [
                row["xmin"],
                row["ymin"],
                row["xmax"],
                row["ymax"]
            ]

            yolo_box = convert_to_yolo(box, img_w, img_h)

            pseudo_labels.append({
                "image_id": i,
                "class": 0, 
                "bbox": yolo_box,
                "score": row["score"]
            })

print("Pseudo-labels geradas:", len(pseudo_labels))

Pseudo-labels geradas: 0


## 3. Gravação das Pseudo-Labels no HDF5

In [14]:
if "labels" in hdf5:
    del hdf5["labels"]

label_group = hdf5.create_group("labels")

image_ids = []
classes = []
bboxes = []
scores = []

for p in pseudo_labels:
    image_ids.append(p["image_id"])
    classes.append(p["class"])
    bboxes.append(p["bbox"])
    scores.append(p["score"])

label_group.create_dataset("image_id", data=np.array(image_ids))
label_group.create_dataset("class", data=np.array(classes))
label_group.create_dataset("bbox", data=np.array(bboxes))
label_group.create_dataset("score", data=np.array(scores))

hdf5.flush()
hdf5.close()

print("Pseudo-labels salvas no HDF5 com sucesso")

Pseudo-labels salvas no HDF5 com sucesso
